# 🎱 Railbird AI - High-Resolution Pool Ball Detector Training (Google Colab)
### Model: `YOLO26s` | Resolution: `1280x1280` | Dataset: `kevinstotz/pool-ball-detection` (Hugging Face)

This Google Colab notebook automates the end-to-end pipeline for training a state-of-the-art pool ball detection model:
1. **Dataset Ingestion**: Streams & downloads `kevinstotz/pool-ball-detection` from Hugging Face Datasets.
2. **YOLO Conversion**: Converts COCO-style pixel bounding boxes `[x, y, w, h]` into normalized YOLO format `[class, x_center, y_center, width, height]` across train, validation, and test splits.
3. **High-Resolution Fine-Tuning**: Trains `yolo26s.pt` at high resolution (`imgsz=1280`, `epochs=80`, `batch=16`) on a Google Colab T4 or L4 GPU.
4. **Validation & Benchmarking**: Evaluates precision, recall, and `mAP50` / `mAP50-95`.
5. **Artifact Export**: Exports and triggers automatic download of the optimized checkpoint `best_pool_1280.pt`.

> 💡 **Tip:** Ensure GPU acceleration is enabled: Go to **Runtime** > **Change runtime type** > Select **T4 GPU** or **L4 GPU**.

In [ ]:
# Step 1: Install required packages
!pip install -q "ultralytics>=8.4.0" "datasets>=2.18.0" huggingface_hub pyyaml pillow tqdm

In [ ]:
# Step 2: Verify GPU availability and PyTorch CUDA backend
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name:     {torch.cuda.get_device_name(0)}")
    print(f"VRAM:            {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    !nvidia-smi
else:
    print("WARNING: CUDA is not available. Please switch runtime to T4/L4 GPU in Colab Runtime settings.")

## 📥 Download & Convert `kevinstotz/pool-ball-detection` to YOLO Format

The dataset on Hugging Face provides pool table images with bounding boxes stored in COCO format: `[x_min, y_min, width, height]` in raw pixels.
We convert them to normalized YOLO format:
$$\text{center}_x = \frac{x_{\min} + \frac{w}{2}}{W}, \quad \text{center}_y = \frac{y_{\min} + \frac{h}{2}}{H}, \quad \text{width} = \frac{w}{W}, \quad \text{height} = \frac{h}{H}$$
All 16 pool ball classes (Cue Ball and 1 to 15 object balls) are mapped and indexed.

In [ ]:
import os
import yaml
from pathlib import Path
from tqdm.auto import tqdm
from datasets import load_dataset
from PIL import Image

# 1. Output directory structure for YOLO
DATASET_ROOT = Path("/content/pool_ball_dataset")
for split in ["train", "val", "test"]:
    (DATASET_ROOT / "images" / split).mkdir(parents=True, exist_ok=True)
    (DATASET_ROOT / "labels" / split).mkdir(parents=True, exist_ok=True)

# 2. Download from Hugging Face
print("Downloading kevinstotz/pool-ball-detection from Hugging Face...")
hf_dataset = load_dataset("kevinstotz/pool-ball-detection")
print(f"Dataset splits loaded: {list(hf_dataset.keys())}")

# Map Hugging Face split names to YOLO split names
split_map = {
    "train": "train",
    "validation": "val",
    "test": "test"
}

# 16 standard billiard ball classes
CLASS_NAMES = {
    0: "cue_ball",
    1: "1_solid_yellow",
    2: "2_solid_blue",
    3: "3_solid_red",
    4: "4_solid_purple",
    5: "5_solid_orange",
    6: "6_solid_green",
    7: "7_solid_brown",
    8: "8_solid_black",
    9: "9_stripe_yellow",
    10: "10_stripe_blue",
    11: "11_stripe_red",
    12: "12_stripe_purple",
    13: "13_stripe_orange",
    14: "14_stripe_green",
    15: "15_stripe_brown"
}

# 3. Process each split
for hf_split, yolo_split in split_map.items():
    if hf_split not in hf_dataset:
        print(f"Skipping missing split: {hf_split}")
        continue

    split_data = hf_dataset[hf_split]
    print(f"\nConverting '{hf_split}' -> '{yolo_split}' ({len(split_data)} images)...")

    img_out_dir = DATASET_ROOT / "images" / yolo_split
    lbl_out_dir = DATASET_ROOT / "labels" / yolo_split

    for idx, item in enumerate(tqdm(split_data, desc=f"Converting {yolo_split}")):
        img = item["image"]
        w_img, h_img = img.size
        sample_id = item.get("image_id", idx)
        file_stem = f"{yolo_split}_{sample_id:06d}"

        # Save image
        img_path = img_out_dir / f"{file_stem}.jpg"
        img.convert("RGB").save(img_path, quality=95)

        # Parse objects (bounding boxes and categories)
        objects = item.get("objects", {})
        bboxes = objects.get("bbox", [])
        categories = objects.get("category", [])

        yolo_lines = []
        for box, cat in zip(bboxes, categories):
            # Hugging Face format: [x_min, y_min, box_w, box_h] in pixels
            x_min, y_min, bw, bh = box
            
            # Normalize to YOLO format: center_x, center_y, width, height (0..1)
            cx = (x_min + bw / 2.0) / float(w_img)
            cy = (y_min + bh / 2.0) / float(h_img)
            norm_w = bw / float(w_img)
            norm_h = bh / float(h_img)

            # Clamp coordinates to [0, 1]
            cx = max(0.0, min(1.0, cx))
            cy = max(0.0, min(1.0, cy))
            norm_w = max(0.0, min(1.0, norm_w))
            norm_h = max(0.0, min(1.0, norm_h))

            cat_id = int(cat)
            yolo_lines.append(f"{cat_id} {cx:.6f} {cy:.6f} {norm_w:.6f} {norm_h:.6f}")

        # Write label file
        lbl_path = lbl_out_dir / f"{file_stem}.txt"
        with open(lbl_path, "w") as f:
            f.write("\n".join(yolo_lines) + "\n")

# 4. Generate data.yaml configuration
yaml_path = DATASET_ROOT / "data.yaml"
data_yaml_content = {
    "path": str(DATASET_ROOT),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "names": CLASS_NAMES
}

with open(yaml_path, "w") as f:
    yaml.dump(data_yaml_content, f, sort_keys=False)

print(f"\n[DONE] YOLO Dataset ready at: {DATASET_ROOT}")
print(f"data.yaml configured at: {yaml_path}")

## 🚀 Fine-Tune `YOLO26s` at 1280px Resolution

We train the model with high-resolution inputs (`imgsz=1280`) to ensure accurate detection of small billiard balls across the entire 9-foot table.
- **Base Architecture**: `yolo26s.pt` (or `yolo11s.pt` / `yolov8s.pt` fallback)
- **Image Size**: `1280`
- **Batch Size**: `16` (fits 16GB Colab T4/L4 VRAM)
- **Epochs**: `80`
- **Optimizer**: `auto` (SGD / AdamW with cosine LR scheduler)

In [ ]:
from ultralytics import YOLO

# Load base model checkpoint
# Defaults to yolo26s.pt; falls back gracefully to yolo11s.pt if pre-release weight naming requires
weights_candidate = "yolo26s.pt"
try:
    model = YOLO(weights_candidate)
except Exception as e:
    print(f"Could not load {weights_candidate} ({e}), falling back to yolo11s.pt")
    weights_candidate = "yolo11s.pt"
    model = YOLO(weights_candidate)

print(f"Loaded base model: {weights_candidate}")

# Launch fine-tuning at imgsz=1280
train_results = model.train(
    data=str(DATASET_ROOT / "data.yaml"),
    epochs=80,
    imgsz=1280,
    batch=16,
    device=0,
    workers=4,
    project="/content/runs/train",
    name="pool_1280",
    exist_ok=True,
    patience=20,
    save=True,
    plots=True,
    verbose=True
)

## 📊 Validation & Benchmark Metrics (`mAP50`, `mAP50-95`)

Validate the fine-tuned model checkpoint on the hold-out validation set at full 1280px resolution.

In [ ]:
from ultralytics import YOLO

best_model_path = "/content/runs/train/pool_1280/weights/best.pt"
val_model = YOLO(best_model_path)

# Run validation on the val split
val_metrics = val_model.val(
    data=str(DATASET_ROOT / "data.yaml"),
    imgsz=1280,
    batch=16,
    device=0,
    split="val"
)

print("\n" + "=" * 55)
print("🎯 FINAL VALIDATION METRICS (1280px)")
print("=" * 55)
print(f"mAP@50:      {val_metrics.box.map50:.4f} ({val_metrics.box.map50*100:.2f}%)")
print(f"mAP@50-95:   {val_metrics.box.map:.4f} ({val_metrics.box.map*100:.2f}%)")
print(f"Precision:   {val_metrics.box.mp:.4f}")
print(f"Recall:      {val_metrics.box.mr:.4f}")
print("=" * 55)

## 💾 Export & Download Fine-Tuned Weights (`best_pool_1280.pt`)

Save the best model checkpoint as `best_pool_1280.pt` and download it to your local machine.

In [ ]:
import shutil
import os

source_best = "/content/runs/train/pool_1280/weights/best.pt"
target_best = "/content/best_pool_1280.pt"

if os.path.exists(source_best):
    shutil.copyfile(source_best, target_best)
    print(f"Copied best model to: {target_best} ({os.path.getsize(target_best) / 1e6:.2f} MB)")
    
    # Trigger download in Google Colab environment
    try:
        from google.colab import files
        print("Initiating automatic download in browser...")
        files.download(target_best)
    except ImportError:
        print(f"Weights saved at {target_best}. Download manually if running outside Google Colab.")
else:
    print(f"Error: {source_best} not found!")